<img src="./img/data_table_logo.png" align="right" width="150">

# data.tables in R
### Dietmar Hareter und Fabian Pribahsnik

### 22.04.2026

## Agenda

### 1. Einführung
### 2. Datein einlesen und ausgeben
### 3. Daten filtern, sortieren, bearbeiten und gruppieren
### 4. Daten verknüpfen
### 5. Ausblick

***


## 1. Einführung

Neben dem bereits vorgestellten Eco-System `tidyverse` erfreut sich auch das Paket `data.table` großer Beliebtheit, insbesondere wenn es um das Bearbeiten großer Datenmengen geht. Im Kern sind `data.tables` verbesserte Versionen von standard `data.frames` die in der Basisversion von R verwendet werden, jedoch arbeiten sie schneller und speichereffizienter. Darüber hinaus bietet das Paket zusätzliche Funktionen zum Einlesen und Ausgeben tabellarischer Daten und stellt durch eine möglichst geringe Mindestanforderung an die R-Version, aktuell Version R 3.5.0 aus dem Jahr 2018, eine langfristige Abwärtskompatibilität sicher.

### 1.1 Große Datenmengen

Der Begriff "große Datenmengen" ist in der Literatur nicht klar abgegrenzt. Für dieses Seminar definieren wir große Datenmengen in mehreren Dimensionen um die vielfältigsten Anwendungsfälle abdecken zu können. Daher beziehen wir uns auf Situationen in denen Daten mit herkömmlichen Datenverarbeitungstools, insbesondere Kalkulationstabellen, nicht einfach verwaltet bzw. analysiert werden können. Verschiedene Metriken dafür könnten beispielsweise sein: 

 - 100+ Mio. Datenpunkte 
 - 1,048,570 Zeilen (Grenze von Excel)
 - 100+ Dateien die kombiniert werden müssen

Referenz: [https://www.oracle.com/de/big-data/what-is-big-data/](https://www.oracle.com/de/big-data/what-is-big-data/)


### 1.2 Performance

Das Paket `data.table` eigenet sich für skalierbare Ressourcen und zeichnet sich insbesondere aus durch:

- **Low-Level-Parallelisierung:** Viele Standardoperationen sind intern so optimiert, dass sie automatisch mehrere CPU-Threads parallel nutzen.
- **Schnelle und skalierbare Aggregationen** Effiziente Verarbeitung extrem großer Datenmengen, z. B. 100 GB im RAM (siehe Benchmarks).

Die gezeigten Benchmarks sind eine Momentaufnahme und nicht representativ, zeigen aber einen nicht-linearen Anstieg der Verarbeitungszeit bei wachsender Dateigröße. Das kann mehrere Gründe haben:

1. `data.table` versucht beim Einlesevorgang den korrekten Datentyp jeder Spalte zu ermitteln. Die Ermittlung der Datentypen kann mit steigeneder Anzahl von Datenpunkten aufwendiger werden.
2. Die Vergleichsmessungen wurden auf einer Maschine durchgeführt die mit anderen Benutzern geteilt wurde. Es kann daher nicht garantiert werden, dass immer die selben Ressourcen (CPU, RAM, ...) bei jedem Testlauf zur Verfügung stehen.
   
<div style="text-align: center;">
    <img src="./img/f_read_timings_01.png" width="700">
</div>


Das Ausgeben/Speichern von Daten im Vergleich zur Einlese von Daten kann im Allgemeinen schneller sein, da:

1. Datentypen bei der Ausgabe bereits bekannt sind und nicht wie bei der Einlese erst ermittelt werden müssen. 
2. Daten müssen beim Einlesen geparst/analysiert werden, d.h. beispielsweise Spalten nach ";" getrennt werden. Dieser Vorgang entfällt bei der Ausgabe von Daten.
   
<div style="text-align: center;">
    <img src="./img/f_write_timings.png" width="700">
</div>


Auch das Zusammenführen vieler kleiner Dateien zu einem Gesamtdatensatz kann mithilfe von data.table sehr effizient gestaltet werden.

<div style="text-align: center;">
    <img src="./img/f_read_timings_02.png" width="600">
</div>

Zusätzlich zu den oben gezeigten, selbst durchgeführten Benchmarks gibt es auch externe Vergleiche. Diese testen beliebte, datenbankähnliche Open-Source-Tools (`DuckDB`, `spark`, `data.table`, `dplyr`, ...) aus dem Data-Science-Bereich über verschiedene Anwendungsszenarien (z. B. Gruppieren, Joins etc.) hinweg. Hier ein Vergleich von `dplyr` (einem Paket aus dem `tidyverse`) zu `data.table` beim Gruppieren von Daten .

<div style="text-align: center;">
    <img src="./img/timings.png" width="700">
</div>

Quelle: [https://duckdblabs.github.io/db-benchmark/](https://duckdblabs.github.io/db-benchmark/)

### 1.3 Modify in Place

In [ ]:
# Lade alle Pakete die für dieses Notebook notwendig sind
library(data.table)
options(  datatable.print.nrows = 15 # The total number of rows to print before the topn logic is triggered.
        , datatable.print.topn = 3)  # When a data.table is printed, only the first topn and last topn rows are displayed
options(repr.matrix.max.rows = 5)    # Limit HTML table output to 10 rows
library(microbenchmark)

Ein wesentlicher Unterschied zwischen `data.table` und sowohl `tidyverse` als auch `R` liegt in der Logik wie Objekte manipuliert/bearbeitet werden. Letztere verwenden das Prinzip des Copy-on-Modify, währenddessen `data.table` die Modify-in-Place-Methode verwendet.

- **Copy on Modify:** Bei der Manipulation eines Objektes wird das Ergebnis in einem neuen Objekt gespeichert/ausgegeben.
- **Modify in Place:** Bei der Manipulation eines Objektes wird das Objeckt direkt geändert.


In [ ]:
# Beispiel für Copy on Modify
# Daten werden "intern" 2 mal kopiert
df <- data.frame(a = 1:3)
df
tracemem(df)                    # Überwache Speicheradresse dieses Objekts
df$a[1] <- 10

In [ ]:
# Beispiel für Modify in Place
# data.table wird nicht kopiert
dt <- data.table(a = 1:3)
dt
tracemem(dt)                    # Überwache Speicheradresse dieses Objekts
dt[1, a := 10]

### 1.4 Daten

EIOPA stellt aggregierte statistische Daten von Versicherungen und Pensionskassen in der EU bereit. Für Pensionskassen gibt es jährlich auf Länderebene Informationen zu Basisdaten, Bilanzen, Anlagepositionen, Aufwendungen, Mitgliederdaten, Beiträgen, Leistungen und Übertragungen. In den folgenden Abschnitten wollen wir mit einem Teil dieser öffentlich zugänglichen Daten arbeiten. Dazu werden diese eingelesen, gefiltert und verknüpft.

Die folgende Übersicht zeigt einen kurzen Auszug der vorhandenen Daten und ihrer Spalten. Dabei wird ersichtlich, dass die Variablen `Reporting country` und `Reference period` in jeder Datei enthalten sind und somit als Schlüssel zum Verknüpfen der Datensätze dienen können.

<div style="text-align: center;">
    <img src="./img/EIOPA_statistics.png" width="800">
</div>


Quelle: [https://www.eiopa.europa.eu/tools-and-data/occupational-pensions-statistics_en](https://www.eiopa.europa.eu/tools-and-data/occupational-pensions-statistics_en)

Ein konkretes Beispiel für eine solche Datenverknüpfung ist der Aufbau eines Länderspiegels.

<div style="text-align: center;">
    <img src="./img/EIOPA_statistics_02.png" width="800">
</div>

## 2. Datein einlesen und ausgeben

### 2.1 Einlese

Mit Hilfe der Funktion `data.table::fread()` (das f steht für fast), kann man Daten aus txt- oder csv-Dateien einlesen die durch Trennzeichen getrennt sind. Zudem versucht die Funktion das verwendete Trennzeichen sowie die Datentypen der einzelnen Spalten automatisch zu erkennen. Mit ihren 35 verfügbaren Parametern bietet die Funktion eine große Flexibilität beim Einlesen von Daten. In diesem Notebook werden wir jedoch größtenteils auf die Standardeinstellungen zurückgreifen.

**Einlesen einer lokale Datei**

In den folgenden Abschnitten verwenden wir die bereits weiter oben erwähnten, öffentlich zugänglichen Daten von EIOPA, über den europäischen Pensionskassenmarkt.

In [ ]:
# Einlesen der Bilanzdatei
dt_bs <- fread("data/PF_A_BalanceSheet.csv")
print(dt_bs)   # Ausgabe in R-Stil
dt_bs          # Ausgabe in Jupyter-Notebook als HTML-Tabelle

Um einen Überblick über die Struktur des Objekts zu erhalten, bietet sich die Funktion `str()` (structure) an. Wir sehen, dass das Objekt sowohl die Klasse `data.table` als auch die Klasse `data.frame` besitzt. Dadurch lässt sich das Objekt nahtlos als Input für zahlreiche statistische Funktionen nutzen, die ein herkömmliches `data.frame` als Input voraussetzen.

In [ ]:
str(dt_bs)

**Einlese über eine URL**

Die Funktion `fread` erlaubt es auch die Daten direkt über eine URL einzulesen.

In [ ]:
# Einlese der Bilanzdaten direkt über eine URL
dt2 <- fread("https://register.eiopa.europa.eu/_layouts/15/download.aspx?SourceUrl=https://register.eiopa.europa.eu/Publications/Pensions%20Statistics/PF_A_BalanceSheet.csv")
identical(dt_bs,dt2) # püfe ob lokale und externe Daten ident sind

**Selektives Einlesen**

Muss man nur mit Teilmengen von großen Dateien arbeiten und hat nur begrenzte Ressourcen (Arbeitsspeicher) zur Verfügung ist ein vollständiges Laden der Datei oft technisch nicht möglich. In solchen Fällen bietet es sich an bereits während des Einlesevorgangs die Daten mit Hilfe von `grep` (Global Regular Expression Print) zu filtern. Damit ist es möglich nach sogenannten Regular Expressions (reguläre Ausdrücke bzw. Suchmuster) diekt in der Einleseroutine zu filtern. Dieses leistungsstarke Kommandozeilen-Tool ist standardmäßig unter Linux/Unix verfügbar, kann aber auch auf Windows installiert werden. Der `grep`-Befehl ist grundsätzlich wie folgt aufgebaut: 

`grep` `[OPTIONEN]` `MUSTER` `[DATEI]`

<center>

| Option | Beschreibung |
| :---: | :--- |
| **-E** | Das Muster als erweiterte regular expressions interpretieren. |
| **-w** | Wählt nur solche Zeilen aus, deren Treffer aus vollständigen Wörtern bestehen. |
| **-v** | Invertiert das Suchmuster, so dass alle Zeilen ausgewählt werden, die nicht auf MUSTER passen. |
| **...** | ... |

</center>

Weiterführende Informationen: [https://wiki.ubuntuusers.de/grep/](https://wiki.ubuntuusers.de/grep/)


<div style="text-align: center;">
    <img src="./img/grep.png" width="800">
</div>

<div style="text-align: center;">
    <img src="./img/grep_result.png" width="500">
</div>

In [ ]:
# Lese nur Kopfzeile, AT und DE Zeilen ein
dt <- fread('grep -E -w "country|AT|DE" data/PF_A_BalanceSheet.csv')
str(dt)
dt

<div style="text-align: center;">
    <img src="./img/grep_gemini.png" width="750">
</div>

### 2.2 Ausgabe

Mit Hilfe der Funktion `data.table::fwrite()` (das f steht für fast), können wir Daten in Textdateien (bspw. .txt oder .csv) speichern.

In [ ]:
# Schreibe .txt und .csv Datei
fwrite(dt, "~/dt.csv")
fwrite(dt, "~/dt.txt", sep = ";")

***
<div align="center">

#### Daten ein- und ausgeben (`fread` & `fwrite`)

| Aktion | Syntax / Code-Beispiel |
| :--- | :--- |
| Datei schnell einlesen | `fread("datei.csv")` |
| Gefiltert einlesen (mit System-Befehl `grep`) | `fread('grep [OPTIONEN] MUSTER [DATEI]')` |
| Daten in eine Datei schreiben | `fwrite(DT, "datei.csv")` |
| Mit spezifischem Trennzeichen schreiben | `fwrite(DT, "datei.csv", sep = ";")` |

</div>


***

**Aufgabe:** Lesen sie alle Zeilen der Date `data/PF_A_BalanceSheet.csv` ein die nicht das Wort `AT` enthalten.


In [ ]:
# Lösung der Aufgabe
#----------------------
dt_1 <- fread('grep -Ewv  "AT" data/PF_A_BalanceSheet.csv')
dt_2 <- fread('grep -wv   "AT" data/PF_A_BalanceSheet.csv')
dt_3 <- fread('grep -w -v "AT" data/PF_A_BalanceSheet.csv')
dt_4 <- fread('grep -v -w "AT" data/PF_A_BalanceSheet.csv')
str(dt)
identical(dt_1, dt_2) && identical(dt_1, dt_3) && identical(dt_1, dt_4)

***

**Aufgabe:** Erstellen Sie auf Basis der Datei `data/PF_A_MEMBERSDATA.csv` einen neuen Datensatz, der ausschließlich die Einträge der aktiven Mitglieder (Code: R0008) enthält. Lesen Sie dazu die Daten ein und filtern Sie während des Einlesevorgangs bereits alle Einträge heraus die nicht mit dem Code `R0008` übereinstimmen.


In [ ]:
# Lösung der Aufgabe
#----------------------
dt <- fread('grep -E -w "country|R0008" data/PF_A_MEMBERSDATA.csv')
dt

## 3 Daten filtern, sortieren und gruppieren

### 3.1 Syntax

Objekte des Typs `data.frame` und `data.table` eigenen sich dafür Daten, die in tabellarischer Form vorliegen, in einer Tabellenstruktur zu speichern.

<div style="text-align: center;">
    <img src="./img/data_frame_struktur.png" width="600">
</div>

Im Gegensatz zum herkömmlichen `data.frame` erlaubt `data.table` weit mehr als nur das Filtern von Zeilen oder Selektieren von Spalten innerhalb der eckigen Klammern `[...]`. Die Syntax innerhalb `DT[...]` weist eine starke Ähnlichkeit zu SQL auf. Um die Funktionsweise zu verstehen, betrachten wir zunächst die Grundstruktur der `data.table` Syntax:

<div style="text-align: center;">
    <img src="./img/data_table_syntax.png" width="600">
</div>

Referenz: [https://machinelearningplus.com/data-manipulation/datatable-in-r-complete-guide/](https://machinelearningplus.com/data-manipulation/datatable-in-r-complete-guide/)

<center>

| Parameter | Bedeutung |
| :---: | :--- |
| **i** | Welche Zeilen sollen berücksichtigt werden (filtern)? |
| **j** | Was soll getan werden (berechnen, selektieren, modifizieren)? |
| **by** | Wie soll aggregiert werden (gruppieren)? |

</center>

### Filtern mit `i`

Über den Parameter `i` können die Daten gefiltert werden, so dass nur bestimmte Zeilen berücksichtigt werden. Da alle Ausdrücke in `i` im Kontext der `data.table` ausgewertet werden, kann direkt mit den Spaltennamen gearbeitet werden. Spaltennamen sollten nach Möglichkeit **keine** Leerzeichen enthalten. Lässt sich das nicht verhindern müssen Spaltennamen innderhalb der eckigen Klammern `DT[...]` mit einem Backtick (``) versehen werden. Der Parameter `check.names = TRUE` kann genutzt werden, um syntaktisch gültige Spaltennamen zu erzeugen. Dabei werden beispielsweise Leerzeichen und andere Sonderzeichen durch einen Punkt (`.`) ersetzt.

In [ ]:
# Alle Variablen löschen und Bilanzdaten neu einlesen um einen klaren Startpunkt zu haben
rm(list = ls()) 
dt_bs <- fread(  "data/PF_A_BalanceSheet.csv"
               , check.names = TRUE) # remove spaces in header names

In [ ]:
dt_bs[Reporting.country == "AT", ]

Lesen wir nun den Datensatz nochmals ein, ohne den Parameter `check.names = TRUE` zu verwenden so zeigt sich, dass die Spaltennamen dann in `` gesetzt werden müssen.

In [ ]:
dt_bs_spaces <- fread(  "data/PF_A_BalanceSheet.csv")
dt_bs_spaces[`Reporting country` == "AT", ]

Möchte man nach mehreren Bedingungen gleichzeitig filtern, so geht das über logischen Operatoren. Für das logische und verwendet man `&`. Für das logische oder verwendet man `|`.

In [ ]:
dt_bs[Reporting.country == "AT" & Reference.period == 2020 & Item.code == 'R0010', ]

Durch den Einsatz von Klammern lassen sich logische Ausdrücke gezielt gruppieren, was gleichzeitig die Lesbarkeit des Codes erhöht.

In [ ]:
dt_bs[(Reporting.country == "AT" | Reporting.country == "PT") & Reference.period == 2020 & Item.code == 'R0010', ]

Für eine bessere Lesbarkeit empfiehlt es sich außerdem, logische Blöcke auf separate Zeilen aufzuteilen und die Einrückung an den logischen Operatoren auszurichten.

In [ ]:
dt_bs[  (Reporting.country %in% c("AT","PT")) 
      & Reference.period == 2020 
      & Item.code == 'R0010', ]

### Sortieren mit `i`

Eine `data.table` lässt sich mit der Funktion `order()` nach einer oder mehreren Spalten sortieren. Für eine Sortierung in aufsteigender Reihenfolge wird der Name der Spalte direkt angegeben. Für eine Sortierung in absteigender Reihenfolge wird dem Spaltennamen das Zeichen `-` vorangestellt. Die Sortierung erfolgt dabei immer in der Reihenfolge in der die Spaltennamen angegeben werden.

In [ ]:
# Mehrere Spalten aufsteigend sortieren
dt_bs[order(Item.code, Reference.period), ]

In [ ]:
# Absteigende Sortierung durch "-"
dt_bs[order(-Reporting.country, Reference.period), ]

***

<div align="center">

#### Verwendung von `i` in data.table

| Aktion | Syntax / Code-Beispiel |
| :--- | :--- |
| Filtern nach einer Bedingung | `DT[colA > value]` |
| Filtern mit UND-Verknüpfung | `DT[colA > value & colB == "Text"]` |
| Filtern mit ODER-Verknüpfung | <code>DT[colA > value &#124; colB == "Text"]</code> |
| Filtern nach mehreren Werten (`%in%`) | `DT[colA %in% c("Wert1", "Wert2")]` |
| Aufsteigend sortieren | `DT[order(colA)]` |
| Absteigend sortieren | `DT[order(-colA)]` |
| Nach mehreren Spalten sortieren | `DT[order(colA, -colB)]` |

</div>

*** 
**Aufgabe:**

Filtern Sie den Datensatz `dt_bs` so, dass er nur noch die Beobachtungen für ein bestimmtes Land und ein spezifisches Jahr enthält.

In [ ]:
# Lösung der Aufgabe
#----------------------
dt_bs[Reporting.country == "BE" & Reference.period == 2020, ]

### Spalten auswählen mit `j`

Als nächstes wenden wir uns nun dem Parameter `j` zu mit dessen Hilfe man Daten manipulieren, berechnen und selektieren kann. 

<div style="text-align: center;">
    <img src="./img/data_table_syntax.png" width="600">
</div>

Referenz: [https://machinelearningplus.com/data-manipulation/datatable-in-r-complete-guide/](https://machinelearningplus.com/data-manipulation/datatable-in-r-complete-guide/)

<center>

Liefert der Ausdruck in `j` eine Liste zurück, wird jedes Element dieser Liste als Spalte der resultierenden `data.table` interpretiert. Wird nur eine einzige Spalte ausgewählt, kann das Ergebnis wahlweise als Vektor oder weiterhin als `data.table` zurückgegeben werden.

In [ ]:
# 1 Spalte als data.table oder als Vektor
dt_bs[, .(Reporting.country)]   # Ergebnis is eine data.table
dt_bs[, Reporting.country]      # Ergebnis is ein Vektor 

Für die Ausgabe von 2 oder mehr Spalten gibt es verschiedene Möglichkeiten die einzelnen Spaltennamen aufzulisten.

In [ ]:
# Ausgabe mehrerer Spalten
dt_bs[, list(Reporting.country, Reference.period)]
dt_bs[, .(Reporting.country, Reference.period)]     # Präferierte Syntax
dt_bs[, c("Reporting.country", "Reference.period")] 

Spaltennamen können auch beliebig für die Ausgabe angepasst werden.

In [ ]:
# Ausgabe mehrerer Spalten mit neuen Namen (Name ist temporär)
dt_bs[, .(Country = Reporting.country,
          Year = Reference.period)] 

Soll ein zusammenhängender Bereich zwischen 2 Spalten ausgegeben werden, ist eine Auflistung der einzelnen Spalten oft nicht praktikabel. In solchen Situationen bietet sich eine Schreibweise an, die alle Spalten zwischen einer Anfangs- und Endspalte abdeckt. 

*Achtung:* Kommt beispielsweise in den Rohdaten eine Spalte hinzu die in dem angegebenen Bereich liegt kann dies leicht zu unerwünschten Ergebnissen führen.

In [ ]:
# Bereich von Spalten ausgeben
dt_bs[, Reporting.country:Item.name]

Möchte man einzelne Spalten ausschließen geschieht dies mittels `!` oder `-` und der Auflistung der Spalten in einem Vektor. Die Spaltennamen müssen dann in Anführungszeichen gesetzt werden.

In [ ]:
# Spalten nicht auswählen
dt_bs[, !c("Reporting.country", "Item.code")]
dt_bs[, -c("Reporting.country", "Item.code")]


Es ist auch möglich einen Bereich von Spalten zu ignorieren. 

In [ ]:
# Bereich von Spalten nicht ausgeben
dt_bs[, -(Reporting.country:Item.name)]
dt_bs[, !(Reporting.country:Item.name)]

***

**Aufgabe:** Basierend auf `dt_bs` filtern Sie die Daten wie folgt:

1. Total Assets (R0268) Einträge für Österreich aus den Jahren 2023 und 2024.
2. Spalten Reporting.country, Reference.period, Item.name, IORP.type und Value..euro.millions. wobei für die Spalten neue sprechende Namen vergeben werden sollen.

In [ ]:
# Lösung der Aufgabe
#----------------------
dt_bs[  Reporting.country == "AT"
      & Reference.period %in% c(2023, 2024)
      & Item.code == "R0268",
     .( Land    = Reporting.country,
        Jahr    = Reference.period,
        Eintrag = Item.name,
        Typ     = IORP.type,
        Wert    = Value..euro.millions.         
     )]

### Daten bearbeiten mit `j`

Über den Parameter `j` können die Daten auch manipuliert werden, ähnlich wie bei der Funkiton `mutate` aus dem `tidyverse`. Eine neue Variable kann über die Zeichenkette `:=` definiert werden (dem so genannten Walrus Operator).

**Neue Spalten hinzufügen**

In [ ]:
# Erzeuge eine neue Spalte die alle Werte in Mrd. darstellt
dt_bs[, Value_mrd := Value..euro.millions. / 1000]

Da das Datum nur aus Ziffern besteht wurde es beim Einlesen als Integer interpretiert. Um es als Datum zu formatieren, wird es zuerst in Text `as.character` umgewandelt und dann als Datum `as.IDate` formatiert. Damit lässt sich dann die neue Spalte für das Datum erstellen.

In [ ]:
# Erzeuge neue Spalte unter Verwendung von Funktionen
dt_bs[, datum := as.IDate(as.character(Date.of.extraction..yyyymmdd.), format = "%Y%m%d")]
dt_bs[, IORP.type:datum]

In den bis jetzt gezeigten Beispielen wurde immer nur *eine* neue Spalte pro Aufruf angelegt. Sollen mehrere Spalten innerhalb eines Funktionsaufrufs mit Hilfe des Walross-Operators `:=` erstellt werden, ist eine leicht abgeänderte Schreibweise erforderlich.

In [ ]:
# Erzeuge zwei Spalten gleichzeitig mit Hilfe des Walross-Operators
dt_bs[, `:=`(valueMrdGruppe = cut(Value_mrd, breaks=c(0,5,10,15,20,25,30,40,50,1000), ordered=TRUE, include.lowest = TRUE),
             Value          = Value..euro.millions. * 1e6
            )]
dt_bs[, IORP.type:Value]

**Temporäre Spalten berechnen**

Die vorherigen Befehle fügten eine neue Spalte dauerhaft zur bereits existierenden `data.table` hinzu. Über den Parameter `j` können nicht nur neue Spalten hinzugefügt werden, sondern auch Ergebnisse berechnet werden die nicht dauerhaft in der `data.table` gespeichert werden sollen.

In [ ]:
# Ausgabe aller Referenzjahre
dt_bs[, unique(Reference.period)]            # Ausgabe als Vektor
dt_bs[, .(unique(Reference.period))]         # Ausgabe als data.table

Den neu berechneten Spalten kann auch ein temporärer Name für die Ausgabe zugewiesen werden.

In [ ]:
# Ausgabe als data.table mit neuem Spaltennamen
dt_bs[, .(Jahr = unique(Reference.period))]

Werden innerhalb von `j` mehrere Funktionen geschachtelt um ein Ergebnis zu berechnen kann dies schnell unübersichtlich werden. In solchen Fällen bietet es sich an auf den bereits bekannten Pipe-Operator `|>` zurückzugreifen.

In [ ]:
# Anzahl der Länder für die Daten vorliegen (betrachte alle Zeiträume)
dt_bs[, length(unique(Reporting.country))]
dt_bs[, Reporting.country |> unique() |> length()]
dt_bs[, uniqueN(Reporting.country)] # uniqueN ist eine data.table Funktion

# Zählen aller Einträge mit .N
dt_bs[, .N]

Berechnungen können natürlich auch mit Filterungen in `i` kombiniert werden.

In [ ]:
# Anzahl Länder für die Daten in 2020 vorliegen
dt_bs[Reference.period == 2020, Reporting.country |> unique() |> length()]

Bei fehlende Werte in einem Datensatz gilt es besonders vorsichtig zu sein, da es zu unvorhergesehenen Ausgaben kommen kann.

In [ ]:
dt_bs[, .(Mean_normal = mean(Value..euro.millions.),
          Mean_adopt  = mean(Value..euro.millions., na.rm = TRUE))]  # na.rm ... remove NAs


***

**Aufgabe:** Basierend auf `dt_bs` berechnen sie den Mittelwert und die Standardabweichung in Mrd. aller Einträge der Gruppe (20,25]:

1. Filtern Sie die Daten entsprechend der gewünschten Gruppe
2. Verwenden Sie die Befehle `mean` und `sd` um die entsprechende Statistik zu berechnen. Achtung: Fehlende Daten können nicht ausgeschlossen werden.

In [ ]:
# Lösung der Aufgabe
#----------------------
dt_bs[valueMrdGruppe == "(20,25]", .(Mittelwert = mean(Value_mrd, na.rm = TRUE),
                                     Stdabwei   = sd(Value_mrd, na.rm = TRUE))]

**Daten pipen**

Objekte vom Typ `data.table` lassen sich mit Hilfe des Pipe-Operators `|>` verknüpfen, was einen nahtlosen, mehrstufigen Analyseprozess ermöglicht.

In [ ]:
# Verknüpfe data.tables mit dem Pipe Operator
dt_bs[Reporting.country == "AT"] |>
    _[order(-Value_mrd)] |>
    _[, Reporting.country:IORP.type]

***

<div align="center">

#### Verwendung von `j` in data.table

| Aktion | Syntax / Code-Beispiel |
| :--- | :--- |
| Spalten auf `data.table`-Art auswählen | `DT[, .(colA, colB)]` |
| Spalten auf `data.frame`-Art auswählen | `DT[, c("colA", "colB")]` |
| Bereiche von Spalten ausgeben | `DT[, colA:colC]` |
| Spalten ausschließen | `DT[, !c("colA", "colB")]` |
| Bereiche von Spalten ausschließen | `DT[, !(colA:colC)]` |
| Berechnungen auf Spalten durchführen | `DT[, .(sum(colA), mean(colB))]` |
| Bei Bedarf Spaltennamen vergeben | `DT[, .(sA = sum(colA), mB = mean(colB))]` |
| Mit `i` kombinieren (Filtern & Auswerten) | `DT[colA > value, sum(colB)]` |
| Innerhalb von `j` pipen | <code>DT[, colA &#124;> unique() &#124;> length()]</code> |
| Anzahl eindeutiger Werte ermitteln | `DT[, uniqueN(colA)]` |
| Anzahl der Zeilen ermitteln | `DT[, .N]` |
| `data.table` pipen | <code>DT[colA == "AT"] &#124;> _[order(colB)] &#124;> _[, colC:colD]</code> |

</div>

***

**Aufgabe:** Erstellen Sie anhand von `dt_bs` eine Liste der größten 3 europäischen DB-Märkte für das Jahr 2024, gemessen an den Total Assets (R0268). Geben Sie das Land, den Wert in mrd. Euro sowie die Mrd Gruppe aus.

1. Filtern Sie die Daten zuerst basierend auf Jahr, IORP Typ und dem gewünschten Item code.
2. Sortieren Sie die Daten in absteigender Reihenfolge.
3. Wählen Sie die gewünschten Spalten aus.
4. Geben Sie die ersten 3 Treffer aus (`head(3)`)

In [ ]:
# Lösung der Aufgabe
#----------------------
dt_bs[Reference.period == 2024 & IORP.type == "DB" & Item.code == "R0268"] |>
    _[order(-Value_mrd)]|>
    _[, .(Reporting.country, Value_mrd, valueMrdGruppe)] |>
    head(3)

### Daten gruppieren mit `by`

Als nächstes wenden wir uns nun dem Parameter `by` zu mit dessen Hilfe Ergebnisse gruppiert werden können.

<div style="text-align: center;">
    <img src="./img/data_table_syntax.png" width="600">
</div>

Referenz: [https://machinelearningplus.com/data-manipulation/datatable-in-r-complete-guide/](https://machinelearningplus.com/data-manipulation/datatable-in-r-complete-guide/)

<center>

Auf diese Weise lässt sich leicht berechnen, für wie viele Länder in den einzelnen Jahren Daten vorliegen.

In [ ]:
dt_bs[, .(Anzahl = uniqueN(Reporting.country)), by = .(Reference.period)]

Auch statistische Größen pro Gruppe lassen sich so leicht berechnen. 

In [ ]:
dt_bs[Item.code == "R0268", 
      .(Max = max(Value_mrd, na.rm = TRUE),
        Avg = mean(Value_mrd, na.rm = TRUE),
        Median = median(Value_mrd, na.rm = TRUE),
        Anzahl = .N
       ),
      by = .(Reference.period)] 

## 4. Daten verknüpfen

In diesem Abschnitt werden verschiedene länderspezifische Indikatoren zusammengeführt, um einen kleinen Länderspiegel zu erstellen. Wir verwenden dazu 3 Dateien:

- Bilanzdaten (total assets)
- Kostendaten (total expenses)
- Daten über Anwartschaften (aktive members)

In [ ]:
# Einlese von Bilanz-, Kosten- und Mitgliederdaten
dt_bs  <- fread("https://register.eiopa.europa.eu/_layouts/15/download.aspx?SourceUrl=https://register.eiopa.europa.eu/Publications/Pensions%20Statistics/PF_A_BalanceSheet.csv",
                check.names = TRUE)
dt_exp <- fread("https://register.eiopa.europa.eu/_layouts/15/download.aspx?SourceUrl=https://register.eiopa.europa.eu/Publications/Pensions%20Statistics/PF_A_EXPENSES.csv",
               check.names = TRUE)
dt_mem <- fread("https://register.eiopa.europa.eu/_layouts/15/download.aspx?SourceUrl=https://register.eiopa.europa.eu/Publications/Pensions%20Statistics/PF_A_MEMBERSDATA.csv",
               check.names = TRUE)

dt_bs |> head(2)
dt_exp |> head(2)
dt_mem |> head(2)

Zunächst wird die Bilanzdatei aufbereitet, um die Total Assets pro Jahr und Land zu aggregieren.

In [ ]:
bs_filtered <- dt_bs[Item.code == "R0268", 
                     .(Total_assets_mn = sum(Value..euro.millions., na.rm = TRUE)),
                     by = .(Reporting.country, Reference.period)]
bs_filtered

Im darauffolgenden Schritt werden die Total Expenses nach Land und Jahr gruppiert und aufsummiert.

<div style="text-align: center;">
    <img src="./img/Total_expenses.png" width="750">
</div>

In [ ]:
exp_filtered <- dt_exp[Item.code == "R0050", 
                     .(Total_expenses_mn = sum(Value..euro.millions., na.rm = TRUE)),
                     by = .(Reporting.country, Reference.period)]
exp_filtered

Abschließend werden die aktiven Mitglieder (Active Members) je Land und Jahr aggregiert.

<div style="text-align: center;">
    <img src="./img/active_members.png" width="850">
</div>

In [ ]:
mem_filtered <- dt_mem[Item.code == "R0008", 
                     .(Active_members = sum(Value, na.rm = TRUE)),
                     by = .(Reporting.country, Reference.period)]
mem_filtered

Nach der Aufbereitung der Einzeldaten wird der Länderspiegel mithilfe mehrerer Left Joins erstellt. Da vorab nicht geprüft wurde, ob in allen Dateien lückenlos dieselben Länder- und Jahreskombinationen vorliegen, bleiben fehlende Werte aus den angefügten Datensätzen in der Zieltabelle unbesetzt (Missing Values). So wird sichergestellt, dass die Struktur des Bilanzdatensatzs exakt erhalten bleibt.

<div style="text-align: center;">
    <img src="./img/left_join.png" width="850">
</div>

In [ ]:
dt_overview <- 
    bs_filtered |>
        merge(x = _,     # Left join mit Kosten
             y = exp_filtered,
             by = c("Reporting.country", "Reference.period"),
             all.x = TRUE) |>
        merge(
             x = _,      # Left join mit Mitgliedern
             y = mem_filtered,
             by = c("Reporting.country", "Reference.period"),
             all.x = TRUE)
dt_overview

Nach der erfolgreichen Zusammenführung der Daten lassen sich nun beispielsweise Durchschnittswerte berechnen.

In [ ]:
dt_overview[, `:=`(Avg_exp_pro_mem = Total_expenses_mn / Active_members * 1e6)]
dt_overview

***

**Aufgabe:** Fügen Sie zwei neue Indikatoren zum Länderspiegel hinzu.

In [ ]:
# Lösung der Aufgabe
#----------------------
dt_overview[, `:=`(Avg_assets_pro_mem = Total_assets_mn / Active_members * 1e6,
                   Kostenquote = Total_expenses_mn / Total_assets_mn)]                  

## 5. Ausblick

Das Paket `data.table` bietet zahlreiche weitere Möglichkeiten der Datenverarbeitung, die ein breites Spektrum an Anwendungsfällen abdecken. In diesem Abschnitt möchten wir exemplarisch einige dieser Funktionen vorstellen. Der Fokus liegt dabei auf erweiterten Joins, dem Umstrukturieren von Daten (Reshaping) sowie der Nutzung von Lag- und Lead-Funktionen.

### 5.1 Lag und Lead

In `data.table` werden lag und lead komfortabel über die Funktion `shift()` realisiert, um gezielt auf vorherige oder nachfolgende Werte einer Zeile zuzugreifen. Diese Operationen sind essenziell, um zeitliche Vergleiche wie etwa das Jahreswachstum innerhalb bestimmter Gruppen (z. B. Länder) zu ermitteln.

In [ ]:
dt_overview[,
             prev_total_assets := shift(Total_assets_mn, 1, type = "lag")
            ,by = .(Reporting.country)]
dt_overview

### 5.2 Reshaping

Die Funktionen `melt` und `dcast` bieten in `data.table` die Möglichkeit, Datensätze flexibel zwischen dem sogenannten Wide- und Long-Format umzustrukturieren. Während `melt` breite Tabellen in ein kompaktes Schlüssel-Wert-Format überführt, transformiert `dcast` diese langen Datenstrukturen wieder in separate Spalten.

In [ ]:
dt_overview[, Reporting.country:Total_assets_mn] |>
    dcast(
         Reporting.country ~ Reference.period,
         value.var = c("Total_assets_mn")
          )

### 5.3 Rolling Joins

Mit einem Rolling Join in data.table lassen sich Tabellen auch dann verknüpfen, wenn die Schlüsselvariablen (wie etwa Zeitstempel) nicht exakt übereinstimmen, indem automatisch auf den zeitlich nächstgelegenen Wert zurückgegriffen wird. Diese Funktion ist besonders wertvoll für die Analyse unregelmäßiger Zeitreihen, um beispielsweise einem Ereignis aus der Haupttabelle den zuletzt gültigen Zustand aus einer Referenztabelle zuzuordnen.

<div style="text-align: center;">
    <img src="./img/rolling_join.png" width="600">
</div>

Referenzen:
- [https://www.produnis.de/tabletrainer/datatable.html](https://www.produnis.de/tabletrainer/datatable.html)
- [https://github.com/rdatatable/data.table](https://github.com/rdatatable/data.table)
- [https://duckdblabs.github.io/db-benchmark/](https://duckdblabs.github.io/db-benchmark/)
- [https://r-datatable.com/](https://r-datatable.com/)
- [https://raw.githubusercontent.com/rstudio/cheatsheets/master/datatable.pdf](https://raw.githubusercontent.com/rstudio/cheatsheets/master/datatable.pdf)